# MCP threat observatory — first look at the corpus

A worked example of the analysis layer: distributions, check coverage, the
observation/inference split, and a drift demonstration.

The corpus is **synthetic**. It was generated by collecting against
`examples/demo-server.mjs` with different postures, so no third-party server
was contacted. Any number here describes the fixture, not the ecosystem.

In [ ]:
import sys
from pathlib import Path

REPO = next(
    p for p in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent)
    if (p / "examples" / "corpus").is_dir()
)
sys.path.insert(0, str(REPO / "python" / "src"))

from mcpobs_analysis import load_corpus
from mcpobs_analysis.metrics import (
    assertion_counts, auth_posture, check_matrix, confidence_counts,
    findings_by_check, rows, severity_counts, target_rows,
)
from mcpobs_analysis.table import render

snapshots = load_corpus([REPO / "examples" / "corpus"])
print(f"{len(snapshots)} synthetic snapshots, {sum(len(s.findings) for s in snapshots)} findings")

## 1. What each fixture exposes

`?` is not zero: it means the catalog was never obtained because the server
refused unauthenticated access. A record must not turn "we could not look"
into "there is nothing there".

In [ ]:
print(render(
    ["target", "auth", "tools", "findings", "highest"],
    [[r["target"], r["auth"], "?" if r["tools"] is None else r["tools"], r["findings"], r["highest_severity"]]
     for r in sorted(target_rows(snapshots), key=lambda r: r["target"])],
    {2: "right", 3: "right"},
))

## 2. Severity and confidence are different questions

Severity asks "how much does this matter if true". Confidence asks "how well
supported is the claim". A report that collapses them into one number is
unreadable to anyone who has to act on it.

In [ ]:
severities = severity_counts(snapshots)
confidences = confidence_counts(snapshots)
print(render(["severity", "count"], [[k, v] for k, v in severities.items()], {1: "right"}))
print()
print(render(["confidence", "count"], [[k, v] for k, v in confidences.items()], {1: "right"}))

try:
    import matplotlib.pyplot as plt
except ImportError:
    print("\nmatplotlib not installed; install 'mcpobs-analysis[analysis]' for charts")
else:
    fig, axes = plt.subplots(1, 2, figsize=(9, 3))
    axes[0].bar(list(severities), list(severities.values()))
    axes[0].set_title("findings by severity")
    axes[1].bar(list(confidences), list(confidences.values()))
    axes[1].set_title("findings by confidence")
    for ax in axes:
        ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

## 3. Which checks fire, and how hard

A check that never fires on this corpus is either unnecessary or badly
specified. This matrix is the honest scoreboard for the check catalogue.

In [ ]:
matrix = check_matrix(snapshots)
header = ["check", "high", "medium", "low", "info", "total"]
body = []
for check, counts in matrix.items():
    body.append([check, counts["high"], counts["medium"], counts["low"], counts["info"], sum(counts.values())])
print(render(header, body, {i: "right" for i in range(1, 6)}))

## 4. Observed versus inferred

This is the field that makes the record usable months later. Anything marked
`inferred` had to be argued, and the argument is in the record.

In [ ]:
print(render(["assertion", "count"], [[k, v] for k, v in assertion_counts(snapshots).items()], {1: "right"}))
print()
for snapshot in snapshots:
    for finding in snapshot.findings:
        if finding.is_inference:
            print(f"{snapshot.path.stem:<20} {finding.check_id:<10} {finding.confidence:<9} {finding.title}")
            print(f"    observation: {finding.observation[:160]}")
            print(f"    inference:   {(finding.inference or '')[:160]}")

## 5. Drift: how a rug pull would look

Take a clean fixture and flip a single annotation. The description does not
change, so a text diff sees nothing, but the client's approval behaviour does.
Hashing the full tool definition catches it.

In [ ]:
import json, tempfile
from mcpobs_analysis import compare, load_snapshot
from mcpobs_analysis.drift import render_drift

before = load_snapshot(REPO / "examples" / "corpus" / "open-readonly.json")
payload = json.loads(json.dumps(before.raw))
payload["probe"]["tools"][0]["annotations"]["destructiveHint"] = True

with tempfile.NamedTemporaryFile("w", suffix=".json", delete=False) as handle:
    json.dump(payload, handle)
    mutated_path = handle.name

drift = compare(before, load_snapshot(mutated_path))
print(render_drift(drift))

## 6. What this notebook does not show

- Anything about the real MCP ecosystem: the corpus is synthetic.
- Runtime behaviour: no server code was executed.
- Intent: no finding here implies a malicious operator.

Before publishing any number derived from this code, read
`docs/METHODOLOGY.md` (evidence grading) and `docs/ETHICS.md` (authorisation).